# Store delivery forecasting: choosing the model and features

Goal of this notebook: decide **which model, features and hyperparameters** go into the production
model, without leaking future information into the choice.

| Step | What happens |
|---|---|
| 1 | Define the forecast problem and the settings |
| 2 | Build features in one function and test it for leakage |
| 3 | Reserve a final holdout and create rolling-origin CV folds |
| 4 | Cross-validate baselines against default LightGBM / XGBoost |
| 5 | Choose the features (ablation and permutation importance) |
| 6 | Choose the training size (all history against the last N weeks) |
| 7 | Tune hyperparameters |
| 8 | Pick the final model or ensemble, score the holdout once |

The holdout is not used in steps 4 to 7. The target of the experiments is the raw weekly quantity per store (no log transform).

**Business framing.** This notebook covers the store level. The chain it serves:

| Layer | Definition | Status |
|---|---|---|
| **Business KPI** | Worker picking cost per colli | Not measured. A stand-in cost index was used in analysis only. |
| **Decision** | Workers per DC and per day | Not built. Needs colli per worker per shift, shift rules and hiring lead times. |
| **ML target** | Colli per DC per delivery date with P80 and P90 | Built: store and week, then store and date, then store, DC and date, summed per DC and date. |
| **ML metric** | Pooled, per store, DC and weekday, on peak days and holiday weeks, with pinball loss and coverage per DC | Built in the backtest. Not done: a DC by weekday table, DC-level metrics per horizon, monitoring of realised values. |

The quantile follows from the cost. If a missing colli costs *r* times a surplus colli, plan at the quantile
*r / (1 + r)*: P80 means r = 4 and P90 means r = 9. Pinball loss at that quantile equals the cost divided by (1 + r),
so the ML metric is the business KPI up to a constant. The cost ratio is not known yet, so P80 and P90 are placeholders.

## 1. Define the forecast problem

#### Based on the exploration. The notebook is mainly aimed at defining the problem. Testing out different features and finding the most relevant features and model.

- Task is to build a modelling pipeline that forecasts the delivery quantity for every store at
every delivery date with forecast horizon of 8 weeks (assumed the business wants a fresh forecast
for the next 8 weeks every week), and to design how you would bring it to production.

- Since, the delivery schedules are not provided beforehand. An assumption is made these are based on a hard to predict pattern. 

- The goal will be to predict outbound colli per store per week. And to split it across the delivery dates provided that week. 

In [1]:
import warnings
from dataclasses import dataclass

import lightgbm as lgb
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.inspection import permutation_importance
from sklearn.model_selection import ParameterSampler
from sklearn.model_selection import TimeSeriesSplit

seed = 0
max_horizon = 8                # forecast the current week and the next 7 (run Monday 08:00, data up to Monday 00:00)
use_sales_lags = False         # lagged realised sales_index as features (needs the sales feed by Monday 08:00)
fresh_lags_count = 3           # most recent known weeks used as lag features (relative to each row's horizon)
target_col = "delivery_quantity_week"
key = ["store", "Monday_of_week"]
key_h = key + ["horizon"]
set_missing_delivery_quantity_as_zero = True

holdout_weeks_count = 12             # final holdout, scored once in step 8
no_of_folds = 4                    # rolling-origin CV folds
test_weeks_count = 12                # length of each CV test window
n_trials = 20                  # random-search trials per model in step 7
min_group_gain = 0.001         # drop a feature group if removing it improves CV WAPE by this much
min_window_gain = 0.001        # use a shorter training window only if it improves CV WAPE by this much

warnings.filterwarnings("ignore", category=UserWarning)

## 2. Features in one function, plus a leakage test

In [2]:
# read the fabricated data 
df = pd.read_csv("../data/input/store_deliveries.csv")

# fix the datetime stamp
df['Monday_of_week'] = pd.to_datetime(df['Monday_of_week'], utc=True)

# define the different feature columns and types
cat_cols = ["country_store", "store"]

feature_groups = {
    "lags_short": [f"qty_last_{k}" for k in range(1, fresh_lags_count + 1)],   # shifted by the row's horizon
    "lags_long": ["qty_lag_52"],
    "rolling": [f"{stat}_{w}" for w in (4, 13, 52) for stat in ("mean", "std")],
    "calendar": ["week_of_year", "month", "week_sin", "week_cos", "xmas_week", "xmas_1w_before", "xmas_2w_before"],
    "schedule": ["n_days", "days_ratio"],      # planned delivery days of the target week: known in advance
    "horizon": ["horizon"],
    "categorical": cat_cols,
}

if use_sales_lags:   # realised sales of weeks known at the origin (shifted by the row's horizon)
    feature_groups["sales"] = [f"sales_last_{k}" for k in range(1, fresh_lags_count + 1)] + ["sales_mean_4"]

all_features = [f for cols in feature_groups.values() for f in cols]
num_features = [f for g, cols in feature_groups.items() if g != "categorical" for f in cols]

In [3]:
# horizon-dependent features: for a row forecast h weeks ahead only weeks <= t - h are known

def add_horizon_features(base, h):
    out = base.copy()
    out["horizon"] = h

    by_store = out.groupby("store")[target_col]
    # fresher lags: qty_last_1 is the most recent known week (t - h), qty_last_2 the week before, ...
    for k in range(fresh_lags_count):
        out[f"qty_last_{k + 1}"] = by_store.shift(h + k)
    out["qty_lag_52"] = by_store.shift(52)

    # rolling statistics over the weeks known at the origin
    shifted = by_store.shift(h)
    for window in (4, 13, 52):
        r = shifted.groupby(out["store"]).rolling(window, min_periods=max(2, window // 2))
        out[f"mean_{window}"] = r.mean().reset_index(level=0, drop=True)
        out[f"std_{window}"] = r.std().reset_index(level=0, drop=True)

    # lagged realised sales: only weeks known at the origin
    sales_by_store = out.groupby("store")["sales_index"]
    for k in range(fresh_lags_count):
        out[f"sales_last_{k + 1}"] = sales_by_store.shift(h + k)
    r = sales_by_store.shift(h).groupby(out["store"]).rolling(4, min_periods=2)
    out["sales_mean_4"] = r.mean().reset_index(level=0, drop=True)

    # n_days (delivery days of the target week) comes from the planned schedule, so it is known in advance;
    # days_ratio compares it with the store's usual number of delivery days (known weeks only)
    r = out.groupby("store")["n_days"].shift(h).groupby(out["store"]).rolling(13, min_periods=6)
    days_usual = r.mean().reset_index(level=0, drop=True)
    out["days_ratio"] = (out["n_days"] / days_usual).where(days_usual > 0)
    return out


# creating a single function to add the necessary features

def add_features (df):

    df = df.assign(delivery_date=pd.to_datetime(df["delivery_date"], utc=True))
    df_weekly = (df.groupby(key, as_index=False)
                   .agg(**{target_col: ("delivery_quantity", "sum"), "sales_index": ("sales_index", "mean"),
                           "n_days": ("delivery_date", "nunique")}))
    df_weekly["Monday_of_week"] = pd.to_datetime(df_weekly["Monday_of_week"], utc=True)

    # reindexing for ensuring the lags are by weeks
    weeks = pd.date_range(df_weekly["Monday_of_week"].min(), df_weekly["Monday_of_week"].max(), freq="7D")
    idx = pd.MultiIndex.from_product([df_weekly["store"].unique(), weeks], names=key)
    df_weekly_reindexed = df_weekly.set_index(key).reindex(idx).reset_index().sort_values(key).reset_index(drop=True)
    
    # True from the first delivery onward
    if set_missing_delivery_quantity_as_zero:
        first_week = df_weekly_reindexed["store"].map(df_weekly.groupby("store")["Monday_of_week"].min())
        keep = df_weekly_reindexed[target_col].notna() | (df_weekly_reindexed["Monday_of_week"] < first_week)
        for col in (target_col, "n_days"):
            df_weekly_reindexed[col] = df_weekly_reindexed[col].where(keep, 0.0)

    # add countries column
    countries = df[['store','country_store']].drop_duplicates()
    dupes = countries.loc[countries["store"].duplicated(keep=False), "store"].unique()
    if len(dupes):
        raise ValueError(f"stores with more than one country_store: {list(dupes)}")
    df_weekly_reindexed = pd.merge(df_weekly_reindexed, countries)

    # add calendar features - iso calender features chosen because the it is important to consider the Monday or week in which it falls
    iso = df_weekly_reindexed["Monday_of_week"].dt.isocalendar()
    iso_year, iso_week = iso["year"].astype(int), iso["week"].astype(int)

    df_weekly_reindexed["week_of_year"] = iso_week
    df_weekly_reindexed["month"] = df_weekly_reindexed["Monday_of_week"].dt.month
    df_weekly_reindexed["week_sin"] = np.sin(2 * np.pi * iso_week / 52)
    df_weekly_reindexed["week_cos"] = np.cos(2 * np.pi * iso_week / 52)

    # add christmas features: ISO week that contains Dec 25 of each row's ISO year
    christmas_week = (pd.to_datetime({"year": iso_year, "month": 12, "day": 25})
                        .dt.isocalendar()["week"].astype(int))

    df_weekly_reindexed["xmas_week"] = (iso_week == christmas_week).astype(int)
    df_weekly_reindexed["xmas_1w_before"] = (iso_week == christmas_week - 1).astype(int)
    df_weekly_reindexed["xmas_2w_before"] = (iso_week == christmas_week - 2).astype(int)
    
    # one copy of the table per forecast horizon, each with its own lags and rolling statistics
    return pd.concat([add_horizon_features(df_weekly_reindexed, h) for h in range(1, max_horizon + 1)],
                     ignore_index=True)

In [4]:
def check_no_leakage(raw, origin):
    """Features of every row forecast from `origin` (last known week) must not depend on later data."""
    poisoned = raw.copy()
    late = poisoned["Monday_of_week"] > origin
    poisoned.loc[late, ["delivery_quantity", "sales_index"]] *= 1000   # make any use of later weeks visible
    # n_days and the calendar come from the planned schedule (known in advance), so they are not poisoned
    both = add_features(raw).merge(add_features(poisoned), on=key_h, suffixes=("_f", "_p"))
    for h in range(1, max_horizon + 1):
        # target week t is forecast with last known week t - h, so t <= origin + h weeks
        rows = both[(both["horizon"] == h) & (both["Monday_of_week"] <= origin + pd.Timedelta(weeks=h))]
        assert len(rows) > 0
        for col in [f for f in num_features if f != "horizon"]:
            pd.testing.assert_series_equal(rows[f"{col}_f"], rows[f"{col}_p"], check_names=False)


weeks_sorted = df["Monday_of_week"].drop_duplicates().sort_values().reset_index(drop=True)
for q in (0.5, 0.75, 0.9):
    check_no_leakage(df, weeks_sorted.iloc[int(q * len(weeks_sorted))])
print("no-leakage checks passed")

df_features = add_features(df)
df_features.shape

no-leakage checks passed


(83200, 29)

## 3. Holdout and rolling-origin CV folds

In [5]:
# Keep only store-weeks the models can use: drop rows without a target and rows without enough history
# (qty_last_1 / mean_13 missing = roughly the first 6 to 13 weeks of each store, fewer for near horizons)
d = (df_features.dropna(subset=[target_col, "qty_last_1", "mean_13"])
       .sort_values("Monday_of_week")
       .reset_index(drop=True))

all_weeks = d["Monday_of_week"].drop_duplicates().sort_values().reset_index(drop=True)
dev_weeks = all_weeks.iloc[:-holdout_weeks_count]
holdout_weeks = all_weeks.iloc[-holdout_weeks_count:]

dev = d[d["Monday_of_week"].isin(dev_weeks)]
holdout = d[d["Monday_of_week"].isin(holdout_weeks)]
# same max_horizon-week gap as in the CV folds
train_for_holdout = dev[dev["Monday_of_week"].isin(dev_weeks.iloc[:-(max_horizon - 1)])]

tscv = TimeSeriesSplit(n_splits=no_of_folds, test_size=test_weeks_count, gap=max_horizon - 1)
folds = [(dev[dev["Monday_of_week"].isin(dev_weeks.iloc[tr])],
          dev[dev["Monday_of_week"].isin(dev_weeks.iloc[te])])
         for tr, te in tscv.split(dev_weeks)]

In [6]:
for fold in folds:

    print(f"train size: {len(fold[0])} and test size {len(fold[1])}")

train size: 21928 and test size 9600
train size: 31528 and test size 9600
train size: 41128 and test size 9600
train size: 50728 and test size 9600


## 4. Baselines against default models (cross-validated)


In [7]:
def wape(y, p):
    return np.abs(y - p).sum() / np.abs(y).sum()


def bias(y, p):
    # > 0 means over-forecast
    return (p.sum() - y.sum()) / y.sum()


def metrics_by(oof, by):
    """WAPE and bias of out-of-fold predictions, per value of the column `by`."""
    return oof.groupby(by, observed=True).apply(
        lambda g: pd.Series({"WAPE": wape(g["y"], g["pred"]), "bias": bias(g["y"], g["pred"])}),
        include_groups=False)


def fold_metrics(oof):
    return metrics_by(oof, "fold")


def summarize(oofs):
    """One row per model: WAPE averaged over the folds, its spread and the worst fold."""
    rows = {}
    for name, oof in oofs.items():
        fm = fold_metrics(oof)
        rows[name] = {"WAPE mean": fm["WAPE"].mean(), "WAPE std": fm["WAPE"].std(),
                      "WAPE worst fold": fm["WAPE"].max(), "bias mean": fm["bias"].mean()}
    return pd.DataFrame(rows).T.sort_values("WAPE mean")


def segment_table(oofs, by):
    return pd.concat({name: metrics_by(oof, by) for name, oof in oofs.items()}, axis=1)

In [8]:
lgb_params = dict(
    n_estimators=600, learning_rate=0.03, num_leaves=15, min_child_samples=40,
    subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0,
    random_state=seed, verbose=-1)
xgb_params = dict(
    n_estimators=600, learning_rate=0.03, max_depth=5, min_child_weight=10,
    subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
    tree_method="hist", enable_categorical=True, random_state=seed)
default_params = {"lgb": lgb_params, "xgb": xgb_params}
model_classes = {"lgb": lgb.LGBMRegressor, "xgb": xgb.XGBRegressor}


@dataclass
class Fitted:
    model: object
    features: list
    dtypes: dict      # category levels taken from the training rows; unseen values become NaN

    def predict(self, frame):
        return np.clip(self.model.predict(frame.astype(self.dtypes)[self.features]), 0, None)


def fit_model(kind, params, features, train):
    """Fit on the raw weekly quantity."""
    dtypes = {c: pd.CategoricalDtype(sorted(train[c].unique())) for c in cat_cols}
    model = model_classes[kind](**params)
    model.fit(train.astype(dtypes)[features], train[target_col])
    return Fitted(model, features, dtypes)

In [9]:
def make_oof(te, fold, pred):
    """Predictions together with the columns used for the segment tables."""
    fallback = te["mean_52"].fillna(te["mean_13"])
    return pd.DataFrame({
        "fold": fold,
        "Monday_of_week": te["Monday_of_week"].to_numpy(),
        "store": te["store"].to_numpy(),
        "country_store": te["country_store"].to_numpy(),
        "horizon": te["horizon"].to_numpy(),
        "volume": pd.qcut(fallback.rank(method="first"), 3, labels=["low", "mid", "high"]).to_numpy(),
        "has_lag_52": te["qty_lag_52"].notna().to_numpy(),     # False = short history
        "y": te[target_col].to_numpy(),
        "pred": np.asarray(pred, dtype=float),
    })


def run_cv(kind, params, features, folds):
    """Out-of-fold predictions: fit on each fold's training rows, predict its test rows."""
    oofs = []
    for i, (train, test) in enumerate(folds):
        fitted = fit_model(kind, params, features, train)
        oofs.append(make_oof(test, i, fitted.predict(test)))
    return pd.concat(oofs, ignore_index=True)


def baseline_preds(te):
    fb = te["mean_13"]
    return {
        "naive: last known week": te["qty_last_1"].fillna(fb),
        "seasonal: lag_52": te["qty_lag_52"].fillna(fb),
        "trailing mean_13": te["mean_13"],
        "trailing mean_52": te["mean_52"].fillna(fb),
    }


def baseline_oofs(folds):
    parts = {}
    for i, (_, test) in enumerate(folds):
        for name, pred in baseline_preds(test).items():
            parts.setdefault(name, []).append(make_oof(test, i, pred))
    return {name: pd.concat(p, ignore_index=True) for name, p in parts.items()}


def blend(*oofs):
    """Simple average of the predictions of several models."""
    out = oofs[0].copy()
    out["pred"] = np.mean([o["pred"].to_numpy() for o in oofs], axis=0)
    return out

In [10]:
baselines = baseline_oofs(folds)

cv_oof = dict(baselines)
cv_oof["LightGBM (default)"] = run_cv("lgb", lgb_params, all_features, folds)
cv_oof["XGBoost (default)"] = run_cv("xgb", xgb_params, all_features, folds)
cv_oof["avg(LGB, XGB) (default)"] = blend(cv_oof["LightGBM (default)"], cv_oof["XGBoost (default)"])

# same default models, different training objectives (the metric is WAPE, an absolute-error metric)
objectives = {"l1": {"lgb": "l1", "xgb": "reg:absoluteerror"},
              "poisson": {"lgb": "poisson", "xgb": "count:poisson"}}
for label, obj in objectives.items():
    cv_oof[f"LightGBM ({label})"] = run_cv("lgb", {**lgb_params, "objective": obj["lgb"]}, all_features, folds)
    cv_oof[f"XGBoost ({label})"] = run_cv("xgb", {**xgb_params, "objective": obj["xgb"]}, all_features, folds)
    cv_oof[f"avg(LGB, XGB) ({label})"] = blend(cv_oof[f"LightGBM ({label})"], cv_oof[f"XGBoost ({label})"])

display(summarize(cv_oof))

,WAPE mean,WAPE std,WAPE worst fold,bias mean
LightGBM (l1),0.137578,0.019661,0.156565,0.015694
"avg(LGB, XGB) (l1)",0.139610,0.022811,0.161653,0.020519
"avg(LGB, XGB) (poisson)",0.141370,0.025021,0.168642,0.021355
LightGBM (poisson),0.141434,0.025319,0.170720,0.021549
XGBoost (poisson),0.142859,0.025362,0.169073,0.021161
XGBoost (l1),0.144276,0.026331,0.169536,0.025344
LightGBM (default),0.144823,0.027503,0.174532,0.025148
"avg(LGB, XGB) (default)",0.146077,0.027697,0.174287,0.025735
trailing mean_52,0.149599,0.022068,0.168829,0.024386
XGBoost (default),0.150004,0.028434,0.177442,0.026321


In [11]:
# Use the best objective per model (by mean CV WAPE; "l2" is the library default) in steps 5 to 8.
# lgb_params / xgb_params are updated in place, so ablation, window choice, tuning and the holdout all use it.
all_objectives = {"l2": {"lgb": "regression", "xgb": "reg:squarederror"}, **objectives}
model_labels = {"lgb": "LightGBM", "xgb": "XGBoost"}

chosen_objective = {}
for kind, label in model_labels.items():
    scores = {name: fold_metrics(cv_oof[f"{label} ({'default' if name == 'l2' else name})"])["WAPE"].mean()
              for name in all_objectives}
    chosen_objective[kind] = min(scores, key=scores.get)

lgb_params["objective"] = all_objectives[chosen_objective["lgb"]]["lgb"]
xgb_params["objective"] = all_objectives[chosen_objective["xgb"]]["xgb"]
print("objective used from here on:", chosen_objective)

objective used from here on: {'lgb': 'l1', 'xgb': 'poisson'}


In [12]:
# Where the default LightGBM is better or worse than the best simple baseline
best_baseline = summarize(baselines).index[0]
pair = {"LightGBM (default)": cv_oof["LightGBM (default)"], best_baseline: cv_oof[best_baseline]}

for by in ("country_store", "volume", "has_lag_52", "horizon"):
    display(segment_table(pair, by))

LightGBM (default)           trailing mean_52          
                            WAPE      bias             WAPE      bias
country_store                                                        
Country A               0.128939  0.020046         0.133582  0.028039
Country B               0.143150  0.013623         0.156030  0.001649
Country C               0.178900  0.054994         0.199815  0.034793
Country D               0.127240  0.014895         0.124197  0.006974
Country E               0.162580  0.020074         0.162812  0.017307

LightGBM (default)           trailing mean_52          
                     WAPE      bias             WAPE      bias
volume                                                        
high             0.135546  0.035913         0.146528  0.035264
low              0.159809 -0.004190         0.160940 -0.006106
mid              0.148049  0.020819         0.149355  0.004335

LightGBM (default)           trailing mean_52          
                         WAPE      bias             WAPE      bias
has_lag_52                                                        
False                0.176614  0.130373         0.139525  0.056573
True                 0.137514  0.002591         0.152234  0.010366

LightGBM (default)           trailing mean_52          
                      WAPE      bias             WAPE      bias
horizon                                                        
1                 0.142302  0.024326         0.148704  0.015950
2                 0.142967  0.024146         0.148992  0.016528
3                 0.143398  0.023652         0.149457  0.017116
4                 0.143917  0.023459         0.149823  0.017721
5                 0.144339  0.023201         0.150305  0.018314
6                 0.144664  0.023315         0.150842  0.018845
7                 0.145136  0.023597         0.151205  0.019431
8                 0.145016  0.023754         0.151765  0.020036

## 5. Choose the features

In [13]:
def ablation(params, features, folds):
    """LightGBM CV WAPE with each feature group removed in turn."""
    configs = {"all features": features}
    for group, cols in feature_groups.items():
        configs[f"without {group}"] = [f for f in features if f not in cols]
    rows = {}
    for name, feats in configs.items():
        fm = fold_metrics(run_cv("lgb", params, feats, folds))
        rows[name] = {"WAPE mean": fm["WAPE"].mean(), "bias mean": fm["bias"].mean(),
                      **{f"fold {i}": w for i, w in fm["WAPE"].items()}}
    table = pd.DataFrame(rows).T
    table["change vs all"] = table["WAPE mean"] - table.loc["all features", "WAPE mean"]
    return table


abl = ablation(lgb_params, all_features, folds)
display(abl)

,WAPE mean,bias mean,fold 0,fold 1,fold 2,fold 3,change vs all
all features,0.137578,0.015694,0.156565,0.126720,0.151516,0.115510,0.000000
without lags_short,0.136923,0.013707,0.152916,0.127961,0.152728,0.114086,-0.000655
without lags_long,0.138309,0.016183,0.156565,0.126720,0.153554,0.116396,0.000731
without rolling,0.138650,0.014302,0.153347,0.130245,0.154794,0.116213,0.001072
without calendar,0.142212,0.007582,0.144856,0.162076,0.142495,0.119423,0.004635
without schedule,0.144649,0.033708,0.171359,0.126390,0.160817,0.120033,0.007072
without horizon,0.138398,0.015016,0.156564,0.128367,0.153143,0.115515,0.000820
without categorical,0.140280,0.013715,0.150016,0.133550,0.161028,0.116528,0.002703


In [14]:
def permutation_table(params, features, folds):
    """Increase in MAE when a feature is shuffled in the test window, averaged over the folds."""
    scores = []
    for train, test in folds:
        fitted = fit_model("lgb", params, features, train)
        result = permutation_importance(
            fitted.model, test.astype(fitted.dtypes)[features], test[target_col],
            n_repeats=3, random_state=seed, scoring="neg_mean_absolute_error")
        scores.append(pd.Series(result.importances_mean, index=features))
    scores = pd.concat(scores, axis=1)
    return (pd.DataFrame({"mae_increase": scores.mean(axis=1), "std_over_folds": scores.std(axis=1)})
              .sort_values("mae_increase", ascending=False))


display(permutation_table(lgb_params, all_features, folds))

,mae_increase,std_over_folds
store,301.966888,33.159533
mean_13,232.959603,50.654280
n_days,64.802878,21.766216
week_of_year,19.061723,30.150022
mean_4,9.632265,5.696430
days_ratio,1.263891,1.893361
week_sin,0.863811,2.709769
qty_lag_52,0.278094,1.598937
std_52,0.234406,1.160690
country_store,0.140203,0.822541


In [15]:
# Drop a feature group when removing it improves CV WAPE by at least min_group_gain.
# Edit drop_groups by hand if the ablation table tells a different story.
change = abl["change vs all"].drop("all features")
drop_groups = [name.removeprefix("without ") for name, v in change.items() if v <= -min_group_gain]
selected_features = [f for f in all_features if not any(f in feature_groups[g] for g in drop_groups)]

print("dropped groups   :", drop_groups or "none")
print("selected features:", selected_features)

dropped groups   : none
selected features: ['qty_last_1', 'qty_last_2', 'qty_last_3', 'qty_lag_52', 'mean_4', 'std_4', 'mean_13', 'std_13', 'mean_52', 'std_52', 'week_of_year', 'month', 'week_sin', 'week_cos', 'xmas_week', 'xmas_1w_before', 'xmas_2w_before', 'n_days', 'days_ratio', 'horizon', 'country_store', 'store']


## 6. Choose the training size

In [16]:
def last_weeks(frame, weeks):
    """Keep the most recent `weeks` weeks of a training frame (None keeps everything)."""
    if weeks is None:
        return frame
    return frame[frame["Monday_of_week"] > frame["Monday_of_week"].max() - pd.Timedelta(weeks=weeks)]


train_windows = {"all history": None, "52 weeks": 52, "35 weeks": 35, "26 weeks": 26}

rows = {}
for label, weeks in train_windows.items():
    folds_w = [(last_weeks(train, weeks), test) for train, test in folds]
    fm = fold_metrics(run_cv("lgb", lgb_params, selected_features, folds_w))
    rows[label] = {"WAPE mean": fm["WAPE"].mean(), "WAPE std": fm["WAPE"].std(),
                   "train rows (avg)": np.mean([len(train) for train, _ in folds_w]),
                   **{f"fold {i}": w for i, w in fm["WAPE"].items()}}
window_results = pd.DataFrame(rows).T
display(window_results)

,WAPE mean,WAPE std,train rows (avg),fold 0,fold 1,fold 2,fold 3
all history,0.137578,0.019661,36328.00,0.156565,0.126720,0.151516,0.115510
52 weeks,0.138333,0.019757,33897.25,0.156565,0.126720,0.153487,0.116558
35 weeks,0.148451,0.018050,26473.00,0.156565,0.159302,0.156490,0.121447
26 weeks,0.187931,0.047801,20710.75,0.236300,0.176138,0.212698,0.126589


In [17]:
# Start from all history and move to a shorter window only if it is clearly better.
train_label = "all history"
for label in list(train_windows)[1:]:
    if window_results.loc[label, "WAPE mean"] <= window_results.loc[train_label, "WAPE mean"] - min_window_gain:
        train_label = label

train_weeks = train_windows[train_label]
folds_w = [(last_weeks(train, train_weeks), test) for train, test in folds]
print("training window:", train_label)

training window: all history


## 7. Tune the hyperparameters

Random search on the selected features and the chosen training window.

In [18]:
lgb_space = {
    "n_estimators": [200, 400, 600, 800],
    "num_leaves": [7, 15, 31],
    "min_child_samples": [20, 40, 80, 160],
    "colsample_bytree": [0.5, 0.7, 0.9],
    "subsample": [0.7, 0.8, 1.0],
    "reg_lambda": [0.0, 1.0, 5.0, 20.0],
}
xgb_space = {
    "n_estimators": [200, 400, 600, 800],
    "max_depth": [3, 4, 5, 6],
    "min_child_weight": [5, 10, 20, 50],
    "colsample_bytree": [0.5, 0.7, 0.9],
    "subsample": [0.7, 0.8, 1.0],
    "reg_lambda": [1.0, 5.0, 20.0],
}


def tune(kind, space, features, folds, n_trials=n_trials):
    """Random search scored by mean CV WAPE. Returns the best parameters and all trials."""
    results = []
    for sample in ParameterSampler(space, n_trials, random_state=seed):
        fm = fold_metrics(run_cv(kind, {**default_params[kind], **sample}, features, folds))
        results.append((fm["WAPE"].mean(), fm["bias"].mean(), sample))
    results.sort(key=lambda r: r[0])
    table = pd.DataFrame([{**sample, "WAPE": wape_mean, "bias": bias_mean} for wape_mean, bias_mean, sample in results])
    return {**default_params[kind], **results[0][2]}, table


best_params = {}
best_params["lgb"], lgb_trials = tune("lgb", lgb_space, selected_features, folds_w)
best_params["xgb"], xgb_trials = tune("xgb", xgb_space, selected_features, folds_w)

display(lgb_trials.head(5))
display(xgb_trials.head(5))

,subsample,reg_lambda,num_leaves,n_estimators,min_child_samples,colsample_bytree,WAPE,bias
0,0.7,20.0,31,600,40,0.5,0.136968,0.014464
1,0.8,20.0,15,200,40,0.9,0.137062,0.015812
2,0.7,5.0,7,200,20,0.5,0.137222,0.016051
3,0.8,5.0,15,200,40,0.9,0.137232,0.016365
4,0.7,1.0,31,400,40,0.7,0.137351,0.015012


,subsample,reg_lambda,n_estimators,min_child_weight,max_depth,colsample_bytree,WAPE,bias
0,0.7,20.0,200,50,6,0.7,0.135354,0.018583
1,0.8,20.0,200,10,5,0.7,0.138501,0.025819
2,0.7,20.0,800,20,4,0.5,0.139742,0.021334
3,1.0,1.0,200,10,4,0.5,0.139962,0.024687
4,0.8,5.0,400,50,5,0.7,0.140041,0.022283


## 8. Final model or ensemble, scored on the holdout once

In [19]:
tuned = {
    "LightGBM (tuned)": run_cv("lgb", best_params["lgb"], selected_features, folds_w),
    "XGBoost (tuned)": run_cv("xgb", best_params["xgb"], selected_features, folds_w),
}
tuned["avg(LGB, XGB) (tuned)"] = blend(tuned["LightGBM (tuned)"], tuned["XGBoost (tuned)"])
candidates = {"LightGBM (tuned)": ["lgb"], "XGBoost (tuned)": ["xgb"], "avg(LGB, XGB) (tuned)": ["lgb", "xgb"]}

cv_summary = summarize({**baselines, **tuned})
display(cv_summary)

final_name = summarize(tuned).index[0]
print("final choice (lowest mean CV WAPE):", final_name)

,WAPE mean,WAPE std,WAPE worst fold,bias mean
"avg(LGB, XGB) (tuned)",0.134170,0.019245,0.150879,0.016523
XGBoost (tuned),0.135354,0.019029,0.152828,0.018583
LightGBM (tuned),0.136968,0.018856,0.153230,0.014464
trailing mean_52,0.149599,0.022068,0.168829,0.024386
trailing mean_13,0.157758,0.041379,0.210513,0.017278
seasonal: lag_52,0.171131,0.027220,0.207273,0.027394
naive: last known week,0.190447,0.045937,0.257112,0.008329


final choice (lowest mean CV WAPE): avg(LGB, XGB) (tuned)


In [20]:
train_final = last_weeks(train_for_holdout, train_weeks)
fits = {kind: fit_model(kind, best_params[kind], selected_features, train_final) for kind in ("lgb", "xgb")}

ho_oof = {name: make_oof(holdout, "holdout", pred) for name, pred in baseline_preds(holdout).items()}
for name, kinds in candidates.items():
    pred = np.mean([fits[kind].predict(holdout) for kind in kinds], axis=0)
    ho_oof[name] = make_oof(holdout, "holdout", pred)

ho_summary = (summarize(ho_oof)[["WAPE mean", "bias mean"]]
              .rename(columns={"WAPE mean": "WAPE", "bias mean": "bias"}))
display(ho_summary)

,WAPE,bias
LightGBM (tuned),0.113168,-0.004442
"avg(LGB, XGB) (tuned)",0.113560,-0.004261
XGBoost (tuned),0.116815,-0.004079
trailing mean_52,0.124846,0.006935
trailing mean_13,0.130936,-0.013229
seasonal: lag_52,0.156106,-0.004510
naive: last known week,0.163237,0.012848


In [21]:
# Holdout breakdown for the final choice
final_oof = {final_name: ho_oof[final_name]}
for by in ("country_store", "volume", "has_lag_52", "horizon"):
    display(segment_table(final_oof, by))

avg(LGB, XGB) (tuned)          
                               WAPE      bias
country_store                                
Country A                  0.122454 -0.003409
Country B                  0.126121  0.022735
Country C                  0.102841 -0.035151
Country D                  0.099160 -0.001033
Country E                  0.123253 -0.004021

avg(LGB, XGB) (tuned)          
                        WAPE      bias
volume                                
high                0.111633  0.000042
low                 0.117223 -0.011224
mid                 0.114418 -0.006947

avg(LGB, XGB) (tuned)          
                            WAPE      bias
has_lag_52                                
True                     0.11356 -0.004261

avg(LGB, XGB) (tuned)          
                         WAPE      bias
horizon                                
1                    0.110858 -0.002635
2                    0.112096 -0.002384
3                    0.112945 -0.002236
4                    0.113711 -0.003002
5                    0.114517 -0.004027
6                    0.114928 -0.005191
7                    0.114527 -0.006454
8                    0.114901 -0.008156

In [22]:
worst_stores = segment_table(final_oof, "store").sort_values((final_name, "WAPE"), ascending=False)
display(worst_stores.head(10))

avg(LGB, XGB) (tuned)          
                       WAPE      bias
store                                
S0084              0.296190  0.040491
S0020              0.224987  0.000773
S0002              0.224428  0.011435
S0024              0.223486  0.218556
S0073              0.212026  0.113521
S0097              0.194780  0.161816
S0079              0.185969 -0.019572
S0005              0.173667 -0.131380
S0071              0.172129 -0.133141
S0083              0.167364 -0.050069

In [23]:
best_baseline_cv = summarize(baselines)["WAPE mean"].idxmin()
best_baseline_ho = ho_summary.loc[list(baselines), "WAPE"].idxmin()

print(f"Final model     : {final_name}")
print(f"Features        : {selected_features}")
print(f"Training window : {train_label}")
print(f"CV WAPE         : {cv_summary.loc[final_name, 'WAPE mean']:.3f}  "
      f"(best baseline {best_baseline_cv}: {cv_summary.loc[best_baseline_cv, 'WAPE mean']:.3f})")
print(f"Holdout WAPE    : {ho_summary.loc[final_name, 'WAPE']:.3f}  "
      f"(best baseline {best_baseline_ho}: {ho_summary.loc[best_baseline_ho, 'WAPE']:.3f})")
print(f"Holdout bias    : {ho_summary.loc[final_name, 'bias']:+.3f}")
for kind in candidates[final_name]:
    print(f"{kind} params     : { {k: v for k, v in best_params[kind].items() if k not in ('verbose', 'random_state')} }")

Final model     : avg(LGB, XGB) (tuned)
Features        : ['qty_last_1', 'qty_last_2', 'qty_last_3', 'qty_lag_52', 'mean_4', 'std_4', 'mean_13', 'std_13', 'mean_52', 'std_52', 'week_of_year', 'month', 'week_sin', 'week_cos', 'xmas_week', 'xmas_1w_before', 'xmas_2w_before', 'n_days', 'days_ratio', 'horizon', 'country_store', 'store']
Training window : all history
CV WAPE         : 0.134  (best baseline trailing mean_52: 0.150)
Holdout WAPE    : 0.114  (best baseline trailing mean_52: 0.125)
Holdout bias    : -0.004
lgb params     : {'n_estimators': 600, 'learning_rate': 0.03, 'num_leaves': 31, 'min_child_samples': 40, 'subsample': 0.7, 'subsample_freq': 1, 'colsample_bytree': 0.5, 'reg_lambda': 20.0, 'objective': 'l1'}
xgb params     : {'n_estimators': 200, 'learning_rate': 0.03, 'max_depth': 6, 'min_child_weight': 50, 'subsample': 0.7, 'colsample_bytree': 0.7, 'reg_lambda': 20.0, 'tree_method': 'hist', 'enable_categorical': True, 'objective': 'count:poisson'}


In [24]:
def daily_quantities(raw):
    """Quantity per store and delivery date."""
    raw = raw.assign(delivery_date=pd.to_datetime(raw["delivery_date"], utc=True))
    return raw.groupby(["store", "Monday_of_week", "delivery_date"], as_index=False)["delivery_quantity"].sum()


def weekday_shares(daily, until, weeks=26):
    """Average share of a store's weekly quantity that goes out on each weekday, over the `weeks` weeks before `until`."""
    recent = daily[(daily["Monday_of_week"] < until) & (daily["Monday_of_week"] >= until - pd.Timedelta(weeks=weeks))].copy()
    recent["weekday"] = recent["delivery_date"].dt.day_of_week
    week_total = recent.groupby(["store", "Monday_of_week"])["delivery_quantity"].transform("sum")
    recent["share"] = recent["delivery_quantity"] / week_total
    return recent.groupby(["store", "weekday"], as_index=False)["share"].mean()


def forecast_next_weeks(raw, models, schedule, horizon=max_horizon, active_weeks=13):
    """Weekly forecast per store for the `horizon` weeks after the last week in `raw` (it must be a complete week)."""
    assert horizon <= max_horizon, "features are built from data at least `horizon` weeks old, so the horizon cannot exceed max_horizon"
    last = raw["Monday_of_week"].max()
    stores = raw.loc[raw["Monday_of_week"] > last - pd.Timedelta(weeks=active_weeks), ["store", "country_store"]].drop_duplicates()

    # Placeholder rows for the future weeks: one per planned delivery date (the schedule is known in advance),
    # quantity 0 and no sales. Week last + k is forecast with the horizon-k rows only, whose features use
    # weeks <= last, so the placeholders never reach a row that is kept.
    sched = schedule[["store", "delivery_date"]].drop_duplicates().copy()
    sched["delivery_date"] = pd.to_datetime(sched["delivery_date"], utc=True)
    sched["Monday_of_week"] = sched["delivery_date"].dt.normalize() - pd.to_timedelta(sched["delivery_date"].dt.dayofweek, unit="D")
    future_rows = sched.merge(stores, on="store").assign(delivery_quantity=0.0)
    features = add_features(pd.concat([raw, future_rows], ignore_index=True))
    future = features[features["Monday_of_week"] > last].merge(stores[["store"]], on="store")
    weeks_ahead = (future["Monday_of_week"] - last) // pd.Timedelta(weeks=1)
    future = future[(future["horizon"] == weeks_ahead) & (weeks_ahead <= horizon)].copy()

    future["forecast"] = np.mean([m.predict(future) for m in models], axis=0)
    return future[["store", "Monday_of_week", "forecast"]]


def split_by_delivery_date(weekly_forecast, schedule, shares):
    """Split each store's weekly forecast over the dates in `schedule` (columns: store, delivery_date)."""
    s = schedule[["store", "delivery_date"]].drop_duplicates().copy()
    s["delivery_date"] = pd.to_datetime(s["delivery_date"], utc=True)
    s["weekday"] = s["delivery_date"].dt.dayofweek
    s["Monday_of_week"] = s["delivery_date"].dt.normalize() - pd.to_timedelta(s["weekday"], unit="D")

    s = s.merge(shares, on=["store", "weekday"], how="left")
    n_dates = s.groupby(["store", "Monday_of_week"])["delivery_date"].transform("count")
    s["share"] = s["share"].fillna(1 / n_dates)                      # no history for this weekday: equal weight
    s["share"] = s["share"] / s.groupby(["store", "Monday_of_week"])["share"].transform("sum")

    out = s.merge(weekly_forecast, on=["store", "Monday_of_week"])
    out["quantity"] = out["forecast"] * out["share"]
    return out[["store", "delivery_date", "quantity"]]


def assumed_schedule(raw, horizon=max_horizon):
    """Placeholder until the real schedules arrive: repeat last week's delivery dates."""
    last = raw["Monday_of_week"].max()
    dates = raw.loc[raw["Monday_of_week"] == last, ["store", "delivery_date"]].drop_duplicates()
    dates = dates.assign(delivery_date=pd.to_datetime(dates["delivery_date"], utc=True))
    return pd.concat([dates.assign(delivery_date=dates["delivery_date"] + pd.Timedelta(weeks=k))
                      for k in range(1, horizon + 1)], ignore_index=True)


# ---- forecast the next 8 weeks and split per delivery date ---------------------------------

# Final model(s) refit on all usable data (in production no gap is needed: every target up to now is known)
train_all = last_weeks(d, train_weeks)
prod_models = [fit_model(kind, best_params[kind], selected_features, train_all) for kind in candidates[final_name]]

schedule = assumed_schedule(df)          # replace with the real schedule (store, delivery_date): it is planned ahead
weekly_forecast = forecast_next_weeks(df, prod_models, schedule)

daily = daily_quantities(df)
shares = weekday_shares(daily, until=df["Monday_of_week"].max() + pd.Timedelta(weeks=1))

daily_forecast = split_by_delivery_date(weekly_forecast, schedule, shares)

display(weekly_forecast.head(8))
display(daily_forecast.sort_values(["store", "delivery_date"]).head(12))


# ---- how good is the split? (holdout weeks: schedule and daily quantities are known) -------

holdout_start = holdout["Monday_of_week"].min()
shares_ho = weekday_shares(daily, until=holdout_start)
actual = daily[daily["Monday_of_week"] >= holdout_start]

# the holdout predictions are stacked over horizons: use the furthest one (the hardest case) once per store-week
model_weekly = (ho_oof[final_name].query("horizon == @max_horizon")[["store", "Monday_of_week", "pred"]]
                .rename(columns={"pred": "forecast"}))
actual_weekly = (actual.groupby(["store", "Monday_of_week"], as_index=False)["delivery_quantity"].sum()
                       .rename(columns={"delivery_quantity": "forecast"})
                       .merge(model_weekly[["store", "Monday_of_week"]]))          # same store-weeks as the model

variants = {
    "split of the actual weekly totals (split error only)": (actual_weekly, shares_ho),
    "split of the forecast weekly totals (end to end)": (model_weekly, shares_ho),
    "equal split of the forecast weekly totals": (model_weekly, shares_ho.iloc[0:0]),
}
rows = {}
for label, (weekly, share_table) in variants.items():
    split = split_by_delivery_date(weekly, actual[["store", "delivery_date"]], share_table)
    both = split.merge(actual[["store", "delivery_date", "delivery_quantity"]], on=["store", "delivery_date"])
    rows[label] = {"WAPE per store and delivery date": wape(both["delivery_quantity"], both["quantity"]),
                   "bias": bias(both["delivery_quantity"], both["quantity"])}
display(pd.DataFrame(rows).T)

/tmp/ipykernel_174289/3770349804.py:27: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  sched["Monday_of_week"] = sched["delivery_date"].dt.normalize() - pd.to_timedelta(sched["delivery_date"].dt.dayofweek, unit="D")
/tmp/ipykernel_174289/3770349804.py:42: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  s["weekday"] = s["delivery_date"].dt.dayofweek


,store,Monday_of_week,forecast
0,S0001,2026-08-31 00:00:00+00:00,967.274214
8,S0002,2026-08-31 00:00:00+00:00,3136.380243
16,S0003,2026-08-31 00:00:00+00:00,1429.258373
24,S0004,2026-08-31 00:00:00+00:00,3127.182563
32,S0005,2026-08-31 00:00:00+00:00,1874.609959
40,S0006,2026-08-31 00:00:00+00:00,2890.602478
48,S0007,2026-08-31 00:00:00+00:00,908.478591
56,S0008,2026-08-31 00:00:00+00:00,1797.546289


,store,delivery_date,quantity
0,S0001,2026-08-31 00:00:00+00:00,316.709916
1,S0001,2026-09-02 00:00:00+00:00,333.167855
2,S0001,2026-09-04 00:00:00+00:00,317.396443
421,S0001,2026-09-07 00:00:00+00:00,314.057114
422,S0001,2026-09-09 00:00:00+00:00,330.377200
423,S0001,2026-09-11 00:00:00+00:00,314.737891
842,S0001,2026-09-14 00:00:00+00:00,312.300750
843,S0001,2026-09-16 00:00:00+00:00,328.529565
844,S0001,2026-09-18 00:00:00+00:00,312.977719
1263,S0001,2026-09-21 00:00:00+00:00,302.080364


/tmp/ipykernel_174289/3770349804.py:42: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  s["weekday"] = s["delivery_date"].dt.dayofweek
/tmp/ipykernel_174289/3770349804.py:42: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  s["weekday"] = s["delivery_date"].dt.dayofweek
/tmp/ipykernel_174289/3770349804.py:42: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  s["weekday"] = s["delivery_date"].dt.dayofweek


,WAPE per store and delivery date,bias
split of the actual weekly totals (split error only),0.160113,0.000000
split of the forecast weekly totals (end to end),0.196761,-0.008156
equal split of the forecast weekly totals,0.197933,-0.008156
